# Runbook 5.1.b: Cross-site scripting (XSS)

**Skill:** Mitigate Risk · **Maps to:** 5.1.A, 5.1.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** 

**Lesson objective.** Explain how cross-site scripting (XSS) lets an attacker inject and run malicious script in another user's browser, identify the three main XSS variants, and recognize the input-handling mistakes that make an application vulnerable.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def render_comment(username: str, comment: str) -> str
```

#### Input
- `username`: the commenter's display name (untrusted input).
- `comment`: the comment body (untrusted input).

#### Output
A single HTML string, the comment card:
```
<div class="c"><b>USERNAME</b>: COMMENT</div>
```
where `USERNAME` and `COMMENT` are the HTML-encoded forms of the two inputs.

#### Encoding rules
- User values are HTML-encoded with `html.escape(value, quote=True)` semantics:
  `&` → `&amp;`, `<` → `&lt;`, `>` → `&gt;`, `"` → `&quot;`, `'` → `&#x27;`.
- The structural markup the function itself emits (`<div class="c">`, `<b>`, `</b>`, `</div>`)
  is real HTML and is **not** encoded.
- No detection, stripping, or blocklisting of input. Encoding alone is the defense.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only (`html`).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).
user values are encoded rather than concatenated raw.

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
import html  # noqa: F401  (you will need this to fix the function)


def render_comment(username: str, comment: str) -> str:
    """
    Return the HTML card that displays a visitor's comment:

        <div class="c"><b>USERNAME</b>: COMMENT</div>

    VULNERABLE starter: it concatenates the username and comment straight into
    the HTML, so a comment like <script>steal()</script> is sent to the next
    visitor's browser as a real script tag (reflected XSS).

    Fix it with context-aware OUTPUT ENCODING: HTML-encode each user value
    (html.escape(value, quote=True)) before placing it into the template, so the
    browser can never parse user input as markup. Do not strip or blocklist input;
    keep the structural <div>/<b> tags as real HTML.
    """
    return '<div class="c"><b>' + username + "</b>: " + comment + "</div>"


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import html


def test_plain_comment_renders():
    out = impl.render_comment("jordan", "Nice, clear write-up")
    assert "jordan" in out
    assert "Nice, clear write-up" in out


def test_structural_tags_present():
    out = impl.render_comment("jordan", "hello")
    assert '<div class="c">' in out
    assert "<b>" in out and "</b>" in out and "</div>" in out


def test_script_payload_encoded():
    out = impl.render_comment("jordan", "<script>steal()</script>")
    assert "<script>" not in out
    assert "&lt;script&gt;" in out


def test_img_onerror_payload_encoded():
    out = impl.render_comment("jordan", '<img src=x onerror=steal()>')
    assert "<img" not in out
    assert "&lt;img" in out


def test_malicious_username_encoded():
    out = impl.render_comment("<script>evil()</script>", "hi")
    assert "<script>" not in out
    assert "&lt;script&gt;" in out


def test_quotes_and_ampersand_encoded():
    out = impl.render_comment("jordan", 'a "b" \'c\' & d')
    assert "&quot;" in out
    assert "&#x27;" in out
    assert "&amp;" in out


def test_user_value_encoded_exactly():
    payload = '"><svg/onload=steal()>&\''
    out = impl.render_comment("jordan", payload)
    assert html.escape(payload, quote=True) in out
    assert payload not in out


def test_ampersand_not_double_or_raw():
    out = impl.render_comment("a&b", "c")
    assert "&amp;" in out


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it